# GRAMMYs Website Performance Analysis

**Web Analytics & Business Strategy Case Study**

**Author:** Lauren Zhang  
**Tools:** Python, pandas, Plotly, Jupyter Notebook

## Project Overview

In 2022, The Recording Academy split its website into two platforms: GRAMMY.com, focused on music fans and awards content, and RecordingAcademy.com, focused on the organization's professional and industry-facing activities.

This analysis evaluates website traffic, user engagement, and audience demographics to determine whether the split improved website performance and whether The Recording Academy should maintain the two-site strategy.

## Business Question

Should The Recording Academy keep GRAMMY.com and RecordingAcademy.com separate, merge them back together, or pursue an alternative strategy?

#**Data**

I'll be working with two files, `grammys_live_web_analytics.csv` and `ra_live_web_analytics.csv`.

These files will contain the following information:

- **date** - The date the data was confirmed. It is in `yyyy-mm-dd` format.
- **visitors** - The number of users who went on the website on that day.
- **pageviews** - The number of pages that all users viewed on the website.
- **sessions** - The total number of sessions on the website. A session is a group of user interactions with your website that take place within a given time frame. For example a single session can contain multiple page views, events, social interactions.
- **bounced_sessions** - The total number of bounced sessions on the website. A bounced session is when a visitor comes to the website and does not interact with any pages / links and leaves.
- **avg_session_duration_secs** - The average length for all session durations for all users that came to the website that day.
- **awards_week** - A binary flag if the dates align with marketing campaigns before and after the Grammys award ceremony was held. This is the big marketing push to get as many eyeballs watching the event.
- **awards_night** - The actual night that Grammy Awards event was held.

# **Traffic Analysis**


In [1]:
# Import libraries
import pandas as pd
import plotly.express as px

In [2]:
# Read in data of 2 databases
full_df = pd.read_csv('datasets/grammy_live_web_analytics.csv')
rec_academy = pd.read_csv('datasets/ra_live_web_analytics.csv')

### Data Preview

In [3]:
# Preview full_df dataframe
full_df.head()

,date,visitors,pageviews,sessions,bounced_sessions,avg_session_duration_secs,awards_week,awards_night
0,2017-01-01,9611,21407,10196,6490,86,0,0
1,2017-01-02,10752,25658,11350,7055,100,0,0
2,2017-01-03,11425,27062,12215,7569,92,0,0
3,2017-01-04,13098,29189,13852,8929,90,0,0
4,2017-01-05,12234,28288,12990,8105,95,0,0


In [4]:
# Preview rec_academy dataframe
rec_academy.head()

,date,visitors,pageviews,sessions,bounced_sessions,avg_session_duration_secs,awards_week,awards_night
0,2022-02-01,928,2856,1092,591,148,0,0
1,2022-02-02,1329,3233,1490,923,90,0,0
2,2022-02-03,1138,3340,1322,754,127,0,0
3,2022-02-04,811,2552,963,534,142,0,0
4,2022-02-05,541,1530,602,326,111,0,0


### Daily Website Traffic



In [5]:
# Plot a line chart of the visitors on the site
fig = px.line(full_df,
       x='date',
       y='visitors')
fig.update_traces(line_color="black")
fig.update_layout(
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)"
)
fig.show()

**Result**: Traffic spikes align with "show Night" most of the time, but often a little early or late to happen in January or March. Except that trend, there is always a peak on November.

### Key Finding

Website traffic shows substantial day-to-day variation, with exceptionally large spikes around major GRAMMY-related events. This suggests that audience demand is highly event-driven rather than evenly distributed throughout the year.

### Awards Night Traffic Impact

In [6]:
# Average number of visitors on awards nights versus other nights
full_df.groupby('awards_night')['visitors'].mean()

,visitors
awards_night,
0,3.238828e+04
1,1.389590e+06


**Result**: Average website traffic on GRAMMY Awards nights was approximately 43 times higher than on non-awards days, demonstrating how strongly traffic is concentrated around the annual event.

This concentration presents a challenge for maintaining audience engagement throughout the rest of the year.

#**Engagement Analysis**

The Recording Academy split its digital presence across two domains, grammy.com and recordingacademy.com. I separated the data from before the split (when both sites were combined) and after the split (when grammy.com data continued independently). The split happened on February 1, 2022 (`2022-02-01`).

2 new dataframes:

1. `combined_site` contain all data with dates before `2022-02-01`.

2. `grammys` contain all data with dates on or after `2022-02-01`.

In [7]:
# Split the data to separate the full_df into two new dataframes
# One for before the switch of the websites and one for after

combined_site = full_df[full_df['date']< '2022-02-01']
grammys = full_df[full_df['date']>= '2022-02-01']

In [8]:
# .copy() prevents pandas from printing a warning message
combined_site = combined_site.copy()
grammys = grammys.copy()

### Engagement Metrics

A. `pages_per_session` metric: the average number of unique pages a user views before leaving the site.

In [9]:
# Create the `frames` list containing all 3 dataframes
frames = [combined_site, grammys, rec_academy]

In [10]:
# Create the `pages_per_session` column for all 3 dataframes
for df in frames:
    df['pages_per_session'] = df['pageviews'] / df['sessions']

In [11]:
# combined_site graph
px.line(combined_site,
       x='date',
       y='pages_per_session')

In [12]:
# grammys graph
px.line(grammys,
       x='date',
       y='pages_per_session')

In [13]:
# rec_academy graph
px.line(rec_academy,
       x='date',
       y='pages_per_session')

**Result**: After website split, the pages per session metric increase in both websites, indicating higher user engagement.

B. `bounce_rate` metric: percentage of users that come to site, never interact with the page, and leave.

In [14]:
def bounce_rate(dataframe):
    """
    Calculate the overall bounce rate for a website.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        DataFrame containing 'bounced_sessions' and 'sessions'.

    Returns
    -------
    float
        Overall bounce rate.
    """
    return dataframe['bounced_sessions'].sum() / dataframe['sessions'].sum()

In [15]:
# Calculate the Bounce Rate for each site
frame_names = ['combined_site', 'grammys', 'rec_academy']

for name, df in zip(frame_names, frames):
    print(f'The bounce rate of {name} is {bounce_rate(df):.2f}.')

The bounce rate of combined_site is 0.42.
The bounce rate of grammys is 0.40.
The bounce rate of rec_academy is 0.34.


C. `average_time_on_site` metric: average time visitors spend on sites in minutes.

In [16]:
# Calculate average session duration in minutes for each site
for name, df in zip(frame_names, frames):
    average_minutes = df['avg_session_duration_secs'].mean() / 60
    print(f'The average time on site of {name} is {average_minutes:.2f} minutes.')

The average time on site of combined_site is 1.71 minutes.
The average time on site of grammys is 1.38 minutes.
The average time on site of rec_academy is 2.14 minutes.


**Result**: The average time spent on websites has increased after website split, showing higher user engagement

### Key Finding

The website split was associated with stronger engagement across several metrics:

- **Bounce rate:** RecordingAcademy.com had the lowest bounce rate (0.34), compared with GRAMMY.com (0.40) and the pre-split combined site (0.42).
- **Average session duration:** RecordingAcademy.com had the longest average time on site at 2.14 minutes, compared with 1.38 minutes on GRAMMY.com and 1.71 minutes on the combined site.
- **Pages per session:** Both post-split websites showed an increase in pages per session, indicating that visitors explored more content during each visit.

Overall, the results suggest that separating the websites did not reduce engagement and may have allowed each platform to better serve its respective audience.

#**Audience Segmentation**

The `grammys_age_demographics.csv` and `tra_age_demographics.csv` each contain the following information:

- **age_group** - The age group range. e.g. `18-24` are all visitors between the ages of 18 to 24 who come to the site.
- **pct_visitors** - The percentage of all of the websites visitors that come from that specific age group.

In [17]:
# Read in the files
age_grammys = pd.read_csv('datasets/grammys_age_demographics.csv')
age_tra = pd.read_csv('datasets/tra_age_demographics.csv')

In [18]:
# Preview the age_grammys file. the age_tra will look very similar.
age_grammys.head()

,age_group,pct_visitors
0,18-24,27.373210
1,25-34,24.129273
2,35-44,18.717867
3,45-54,13.568619
4,55-64,9.817036


### Combining Audience Data

In [19]:
# Label rows as 'Recording Academy'
age_tra['website'] = 'Recording Academy'

# Label rows as 'Grammys'
age_grammys['website'] = 'Grammys'

In [20]:
# Concatenate dataframes
age_df = pd.concat([age_tra, age_grammys])

# Preview combined data
age_df

,age_group,pct_visitors,website
0,18-24,27.116827,Recording Academy
1,25-34,26.155406,Recording Academy
2,35-44,19.548684,Recording Academy
3,45-54,13.823158,Recording Academy
4,55-64,8.235619,Recording Academy
5,65+,5.120306,Recording Academy
0,18-24,27.373210,Grammys
1,25-34,24.129273,Grammys
2,35-44,18.717867,Grammys
3,45-54,13.568619,Grammys


### Age Distribution by Website

In [21]:
# age_group and pct_visitors bar chart
fig = px.bar(
    age_df,
    x='age_group',
    y='pct_visitors',
    color='website',
    barmode='group',
    title='Visitor Age Distribution by Website',
    labels={
        'age_group': 'Age Group',
        'pct_visitors': 'Percent of Visitors',
        'website': 'Website'
    }
)

fig.show()

**result**: In age 25-44, more people visit Recording Academy. People aged between 55+ visit grammys more. People of 18-24 and 45-54 visit

### Key Finding

The two websites attract somewhat different audience segments:

- Visitors aged **25–44** were more likely to visit RecordingAcademy.com.
- Visitors aged **55+** were more likely to visit GRAMMY.com.
- Visitors aged **18–24** and **45–54** were distributed relatively evenly between the two websites.
- The 55+ segment represented a smaller share of overall visitors, limiting its influence on the total audience distribution.

These differences support maintaining distinct website experiences tailored to different audience needs.

#**Competitive Benchmark: American Music Awards**

To place the GRAMMYs digital strategy in a broader industry context, I compared GRAMMY.com traffic with the American Music Awards (AMA) website.

This benchmark focuses on traffic volume and device usage to identify differences in how audiences access major music awards websites.


**Total Visits**: the total number of visitors on the website during the timespan given.

**Device Distribution**: percentage share of visitors coming from Desktop users (PCs, Macs, etc.) and Mobile Users (iPhone, Android, etc.).

Visitors on the AMA website are spending on average, 5 mins and 53 seconds on the site and viewing 2.74 pages per visit (aka session). They have a bounce rate of 54.31%

In [22]:
# Load in the data
desktop_users = pd.read_csv('datasets/desktop_users.csv')
mobile_users = pd.read_csv('datasets/mobile_users.csv')

In [23]:
# Preview the desktop_users file
desktop_users.head()

,date,segment,visitors
0,2022-02-01,Desktop Traffic,10195
1,2022-02-02,Desktop Traffic,10560
2,2022-02-03,Desktop Traffic,9935
3,2022-02-04,Desktop Traffic,8501
4,2022-02-05,Desktop Traffic,5424


In [24]:
# Preview mobile_users file
mobile_users.head()

,date,segment,visitors
0,2022-02-01,Mobile Traffic,23494
1,2022-02-02,Mobile Traffic,20234
2,2022-02-03,Mobile Traffic,22816
3,2022-02-04,Mobile Traffic,18592
4,2022-02-05,Mobile Traffic,13298


### Combining Competitor Data

In [25]:
# Change name of the visitors column to indicate which category it comes from
desktop_users = desktop_users.rename(columns = {'visitors':'desktop_visitors'})
mobile_users = mobile_users.rename(columns = {'visitors':'mobile_visitors'})

In [26]:
# Drop the segment column from each dataframe
desktop_users = desktop_users.drop(columns=['segment'])
mobile_users = mobile_users.drop(columns = ['segment'])

In [27]:
# Join the two dataframes and preview the dataframe
segment_df = pd.merge(desktop_users, mobile_users, on = 'date', how = 'inner')

### Desktop vs. Mobile Traffic

In [28]:
# Create total_visitors column
segment_df['total_visitors'] = segment_df['desktop_visitors'] + segment_df['mobile_visitors']

In [29]:
# Filter and calculate the percentage share
# Use an f string to print each percentage to the screen
filtered_df = segment_df[segment_df['date']<= '2023-04-01']
percentage = (filtered_df['desktop_visitors'].sum() / filtered_df['total_visitors'].sum())*100
print(f'The percentage share of users coming from desktop is {percentage:.2f}.')
print(f'The percentage share of users coming from mobile is {100-percentage:.2f}.')


The percentage share of users coming from desktop is 26.04.
The percentage share of users coming from mobile is 73.96.


**Result**: The Grammys website has lower bounce rate than its competitor, and the total visits between months are stable, without significant decrease. However, visitors also spend far less average time on the Grammys, and less pages per visists. The Grammys website needs to find ways to attract users to stay on the website for longer time.

### Key Finding

GRAMMY.com attracted substantially more traffic than the AMA website during the observed period. Both websites also received the majority of their traffic from mobile devices, highlighting the importance of a mobile-first user experience for music awards audiences.

The competitive benchmark reinforces the value of optimizing GRAMMY.com specifically for its fan-facing audience rather than treating it as a general-purpose organizational website.

# **Business Recommendation**


Based on the website engagement and demographic data, I recommend that The Recording Academy keep the Recording Academy and GRAMMYs websites separate while continuing to monitor their performance.

The split appears to have improved user engagement. Pages per session increased on both websites after the split, indicating that visitors are exploring more content during each session. Bounce rates are also lower on the separate sites: 0.40 for GRAMMYs and 0.34 for Recording Academy, compared with 0.42 for the combined site. The particularly low bounce rate of the Recording Academy site suggests that its visitors are more likely to continue interacting with the website after arriving. Average time on the website of Recording Academy website is higher, but the average time on the Grammys website is lower after splitting.

The demographic data also supports maintaining separate sites. Visitors aged 25–44 are more likely to visit the Recording Academy site, while visitors aged 55+ are more likely to visit the GRAMMYs site. Visitors aged 18–24 and 45–54 are distributed relatively evenly between the two. Although the 55+ group represents a smaller portion of total visitors, these differences suggest that the two sites attract somewhat distinct audiences. Based on this result, increasing content attractive to its target age group may increase site engagement.

Overall, the increased pages per session and lower bounce rates suggest that separating the websites has improved engagement. The Recording Academy should therefore maintain the two-site structure, target different age group in sites, improve the Grammys website while continuing to track engagement and demographic trends to determine whether these improvements persist over time.